In [ ]:
import numpy as np
import pandas as pd
import torch
import os
import matplotlib.pyplot as plt

In [ ]:
import sys

from flare_preprocessing import *
from utilities import *

In [ ]:
%load_ext autoreload
%reload_ext autoreload

In [ ]:
working_dir = 'D:/2024_S1/ML_SEP_2402/Final_update/Open_Repo'
opr = opr_data_preprocessing(working_dir + "/data/v2_ftp_flares_1997_2024.csv")
sci = sci_data_preprocessing(working_dir + "/data/Sci_matched_with_assigned_ar_20100101_20241231.csv", opr)

In [ ]:
from New_SampleConstruction import *

In [ ]:
from pathlib import Path

def read_all_harp_csvs(root: str | Path, recursive: bool = False) -> pd.DataFrame:
    """
    Read all per-HARP CSVs under `root`, concatenate them in ascending HARP order,
    Assumes files are named like HARP_<harpnum>.csv.
    """

    root = Path(root)
    pattern = "**/HARP_*.csv" if recursive else "HARP_*.csv"

    # List files first, sorted by HARP number
    harp_files = []
    for f in root.glob(pattern):
        try:
            harpnum = int(f.stem.split("_")[1])
            harp_files.append((harpnum, f))
        except Exception:
            continue

    # sort by numeric harp number
    harp_files.sort(key=lambda x: x[0])

    frames = []

    for harpnum, f in harp_files:
        # read with T_REC parsed as datetime
        df = pd.read_csv(f, parse_dates=["T_REC"])
        print(f"Reading HARP {harpnum}")

        # Skip if the HARP lifetime is less than 120 rows (24h / 12min cadence)
        if len(df) < 120:
            print(f"  -> Skipping HARP {harpnum} (too short)")
            continue
        # fill the HARPNUM with the harpnum
        df['HARPNUM'] = harpnum
        frames.append(df)

    if not frames:
        print("No valid HARP CSVs found.")
        return pd.DataFrame()

    # Concatenate in correct order
    full_df = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} HARPs; total rows = {len(full_df)}")
    
    return full_df

In [8]:
nrt_sharps = read_all_harp_csvs(r"D:\\Input Data\\Operational Data\\HMI\SHARP_by_HARP")

Reading HARP 4514
  -> Skipping HARP 4514 (too short)
Reading HARP 4516
  -> Skipping HARP 4516 (too short)
Reading HARP 4518
Reading HARP 4520
  -> Skipping HARP 4520 (too short)
Reading HARP 4522
  -> Skipping HARP 4522 (too short)
Reading HARP 4524
  -> Skipping HARP 4524 (too short)
Reading HARP 4525
Reading HARP 4526
  -> Skipping HARP 4526 (too short)
Reading HARP 4527
Reading HARP 4528
  -> Skipping HARP 4528 (too short)
Reading HARP 4529
  -> Skipping HARP 4529 (too short)
Reading HARP 4530
Reading HARP 4531
  -> Skipping HARP 4531 (too short)
Reading HARP 4532
  -> Skipping HARP 4532 (too short)
Reading HARP 4533
  -> Skipping HARP 4533 (too short)
Reading HARP 4534
Reading HARP 4536
Reading HARP 4538
Reading HARP 4539
  -> Skipping HARP 4539 (too short)
Reading HARP 4540
Reading HARP 4541
  -> Skipping HARP 4541 (too short)
Reading HARP 4543
  -> Skipping HARP 4543 (too short)
Reading HARP 4544
Reading HARP 4546
Reading HARP 4547
Reading HARP 4548
Reading HARP 4550
  -> Skipp

C:\Users\huke0\AppData\Local\Temp\ipykernel_12192\2427577827.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(f, parse_dates=["T_REC"])


Reading HARP 11102
Reading HARP 11103
  -> Skipping HARP 11103 (too short)
Reading HARP 11104
Reading HARP 11105
Reading HARP 11106
  -> Skipping HARP 11106 (too short)
Reading HARP 11107
  -> Skipping HARP 11107 (too short)
Reading HARP 11108
  -> Skipping HARP 11108 (too short)
Reading HARP 11123
  -> Skipping HARP 11123 (too short)
Reading HARP 11172
Reading HARP 11174
Reading HARP 11197
Reading HARP 11199
Reading HARP 11200
Reading HARP 11203
Reading HARP 11207
  -> Skipping HARP 11207 (too short)
Reading HARP 11209
Reading HARP 11210
Reading HARP 11211
  -> Skipping HARP 11211 (too short)
Reading HARP 11212
  -> Skipping HARP 11212 (too short)
Reading HARP 11215
Reading HARP 11217
Reading HARP 11218
Reading HARP 11219
Reading HARP 11220
  -> Skipping HARP 11220 (too short)
Reading HARP 11221
Reading HARP 11222
Reading HARP 11223
  -> Skipping HARP 11223 (too short)
Reading HARP 11224
  -> Skipping HARP 11224 (too short)
Reading HARP 11225
Reading HARP 11226
  -> Skipping HARP 1122

In [9]:
# delete rows with key columns any NaN values
key_columns = [
    'USFLUX','MEANGAM','MEANGBT','MEANGBZ','MEANGBH','MEANJZD',
    'TOTUSJZ','MEANALP','MEANJZH','TOTUSJH','ABSNJZH','SAVNCPP',
    'MEANPOT','TOTPOT','MEANSHR','SHRGT45','SIZE','SIZE_ACR',
    'NACR','NPIX'
]
nrt_sharps = nrt_sharps.dropna(subset=key_columns)

In [10]:
# cut the time range from 2020
sci = sci[sci['peak_time'] >= '2020-01-01']
opr = opr[opr['peak_time'] >= '2020-01-01']
nrt_sharps = nrt_sharps[nrt_sharps['T_REC'] >= '2020-01-01']

In [11]:
nowcast_sci_nrt = New_SampleConstruction()
nowcast_sci_nrt.samples_from_harp(sci, nrt_sharps, lead_window=0)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [12]:
nowcast_opr_nrt = New_SampleConstruction()
nowcast_opr_nrt.samples_from_harp(opr, nrt_sharps, lead_window=0)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [22]:
lead6_sci_nrt = New_SampleConstruction()
lead6_sci_nrt.samples_from_harp(sci, nrt_sharps, lead_window=6)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [23]:
lead6_opr_nrt = New_SampleConstruction()
lead6_opr_nrt.samples_from_harp(opr, nrt_sharps, lead_window=6)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [24]:
lead12_sci_nrt = New_SampleConstruction()
lead12_sci_nrt.samples_from_harp(sci, nrt_sharps, lead_window=12)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [25]:
lead12_opr_nrt = New_SampleConstruction()
lead12_opr_nrt.samples_from_harp(opr, nrt_sharps, lead_window=12)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [26]:
lead24_sci_nrt = New_SampleConstruction()
lead24_sci_nrt.samples_from_harp(sci, nrt_sharps, lead_window=24)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


In [27]:
lead24_opr_nrt = New_SampleConstruction()
lead24_opr_nrt.samples_from_harp(opr, nrt_sharps, lead_window=24)

Processing HARPNUM: 6059
Processing HARPNUM: 6060
Processing HARPNUM: 6061
Processing HARPNUM: 6066
Processing HARPNUM: 6067
Processing HARPNUM: 6086
Processing HARPNUM: 6093
Processing HARPNUM: 6095
Processing HARPNUM: 6098
Processing HARPNUM: 6099
Processing HARPNUM: 6101
Processing HARPNUM: 6102
Processing HARPNUM: 6103
Processing HARPNUM: 6104
Processing HARPNUM: 6110
Processing HARPNUM: 6118
Processing HARPNUM: 6121
Processing HARPNUM: 6122
Processing HARPNUM: 6127
Processing HARPNUM: 6130
Processing HARPNUM: 6132
Processing HARPNUM: 6133
Processing HARPNUM: 6136
Processing HARPNUM: 6139
Processing HARPNUM: 6143
Processing HARPNUM: 6147
Processing HARPNUM: 6148
Processing HARPNUM: 6153
Processing HARPNUM: 6156
Processing HARPNUM: 6162
Processing HARPNUM: 6164
Processing HARPNUM: 6167
Processing HARPNUM: 6174
Processing HARPNUM: 6176
Processing HARPNUM: 6177
Processing HARPNUM: 6178
Processing HARPNUM: 6181
Processing HARPNUM: 6182
Processing HARPNUM: 6183
Processing HARPNUM: 6184


## Results

### LSTM + NRT

In [16]:
import importlib
from Gen_Results import GenMetrics_lstm

#importlib.reload(Gen_Results)

In [17]:
period_results = GenMetrics_lstm(
    model_name="nowcast_Task1_sci_nrt",
    sample_obj=nowcast_sci_nrt,
    model_dir="./LSTM_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-S-NRT-0.csv",
)

Threshold: 0.512[0.512,0.512]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 602
Bootstrap 1/30: nowcast_Task1_sci_nrt0_0.5116666666666666.pth, threshold=0.511667
Bootstrap 2/30: nowcast_Task1_sci_nrt1_0.5116666666666666.pth, threshold=0.511667
Bootstrap 3/30: nowcast_Task1_sci_nrt2_0.5116666666666666.pth, threshold=0.511667
Bootstrap 4/30: nowcast_Task1_sci_nrt3_0.5116666666666666.pth, threshold=0.511667
Bootstrap 5/30: nowcast_Task1_sci_nrt4_0.5116666666666666.pth, threshold=0.511667
Bootstrap 6/30: nowcast_Task1_sci_nrt5_0.5116666666666666.pth, threshold=0.511667
Bootstrap 7/30: nowcast_Task1_sci_nrt6_0.5116666666666666.pth, threshold=0.511667
Bootstrap 8/30: nowcast_Task1_sci_nrt7_0.5116666666666666.pth, threshold=0.511667
Bootstrap 9/30: nowcast_Task1_sci_nrt8_0.5116666666666666.pth, threshold=0.511667
Bootstrap 10/30: nowcast_Task1_sci_nrt9_0.5116666666666666.pth, threshold=0.511667
Bootstrap 11/30: nowcast_Task1_sci_nrt10_0.5116666666666666.pth, threshold=

In [18]:
period_results = GenMetrics_lstm(
    model_name="nowcast_Task2_sci_nrt",
    sample_obj=nowcast_sci_nrt,
    model_dir="./LSTM_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-S-NRT-0.csv",
)

Threshold: 0.410[0.410,0.410]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 879
Bootstrap 1/30: nowcast_Task2_sci_nrt0_0.41.pth, threshold=0.41
Bootstrap 2/30: nowcast_Task2_sci_nrt1_0.41.pth, threshold=0.41
Bootstrap 3/30: nowcast_Task2_sci_nrt2_0.41.pth, threshold=0.41
Bootstrap 4/30: nowcast_Task2_sci_nrt3_0.41.pth, threshold=0.41
Bootstrap 5/30: nowcast_Task2_sci_nrt4_0.41.pth, threshold=0.41
Bootstrap 6/30: nowcast_Task2_sci_nrt5_0.41.pth, threshold=0.41
Bootstrap 7/30: nowcast_Task2_sci_nrt6_0.41.pth, threshold=0.41
Bootstrap 8/30: nowcast_Task2_sci_nrt7_0.41.pth, threshold=0.41
Bootstrap 9/30: nowcast_Task2_sci_nrt8_0.41.pth, threshold=0.41
Bootstrap 10/30: nowcast_Task2_sci_nrt9_0.41.pth, threshold=0.41
Bootstrap 11/30: nowcast_Task2_sci_nrt10_0.41.pth, threshold=0.41
Bootstrap 12/30: nowcast_Task2_sci_nrt11_0.41.pth, threshold=0.41
Bootstrap 13/30: nowcast_Task2_sci_nrt12_0.41.pth, threshold=0.41
Bootstrap 14/30: nowcast_Task2_sci_nrt13_0.41.pth, thresh

In [29]:
period_results = GenMetrics_lstm(
    model_name="nowcast_Task1_opr_nrt",
    sample_obj=nowcast_sci_nrt,
    model_dir="./LSTM_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-O-NRT-0.csv",
)

Threshold: 0.610[0.610,0.610]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 602
Bootstrap 1/30: nowcast_Task1_opr_nrt0_0.61.pth, threshold=0.61
Bootstrap 2/30: nowcast_Task1_opr_nrt1_0.61.pth, threshold=0.61
Bootstrap 3/30: nowcast_Task1_opr_nrt2_0.61.pth, threshold=0.61
Bootstrap 4/30: nowcast_Task1_opr_nrt3_0.61.pth, threshold=0.61
Bootstrap 5/30: nowcast_Task1_opr_nrt4_0.61.pth, threshold=0.61
Bootstrap 6/30: nowcast_Task1_opr_nrt5_0.61.pth, threshold=0.61
Bootstrap 7/30: nowcast_Task1_opr_nrt6_0.61.pth, threshold=0.61
Bootstrap 8/30: nowcast_Task1_opr_nrt7_0.61.pth, threshold=0.61
Bootstrap 9/30: nowcast_Task1_opr_nrt8_0.61.pth, threshold=0.61
Bootstrap 10/30: nowcast_Task1_opr_nrt9_0.61.pth, threshold=0.61
Bootstrap 11/30: nowcast_Task1_opr_nrt10_0.61.pth, threshold=0.61
Bootstrap 12/30: nowcast_Task1_opr_nrt11_0.61.pth, threshold=0.61
Bootstrap 13/30: nowcast_Task1_opr_nrt12_0.61.pth, threshold=0.61
Bootstrap 14/30: nowcast_Task1_opr_nrt13_0.61.pth, thresh

In [30]:
period_results = GenMetrics_lstm(
    model_name="nowcast_Task2_opr_nrt",
    sample_obj=nowcast_sci_nrt,
    model_dir="./LSTM_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-O-NRT-0.csv",
)

Threshold: 0.490[0.490,0.490]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 879
Bootstrap 1/30: nowcast_Task2_opr_nrt0_0.49.pth, threshold=0.49
Bootstrap 2/30: nowcast_Task2_opr_nrt1_0.49.pth, threshold=0.49
Bootstrap 3/30: nowcast_Task2_opr_nrt2_0.49.pth, threshold=0.49
Bootstrap 4/30: nowcast_Task2_opr_nrt3_0.49.pth, threshold=0.49
Bootstrap 5/30: nowcast_Task2_opr_nrt4_0.49.pth, threshold=0.49
Bootstrap 6/30: nowcast_Task2_opr_nrt5_0.49.pth, threshold=0.49
Bootstrap 7/30: nowcast_Task2_opr_nrt6_0.49.pth, threshold=0.49
Bootstrap 8/30: nowcast_Task2_opr_nrt7_0.49.pth, threshold=0.49
Bootstrap 9/30: nowcast_Task2_opr_nrt8_0.49.pth, threshold=0.49
Bootstrap 10/30: nowcast_Task2_opr_nrt9_0.49.pth, threshold=0.49
Bootstrap 11/30: nowcast_Task2_opr_nrt10_0.49.pth, threshold=0.49
Bootstrap 12/30: nowcast_Task2_opr_nrt11_0.49.pth, threshold=0.49
Bootstrap 13/30: nowcast_Task2_opr_nrt12_0.49.pth, threshold=0.49
Bootstrap 14/30: nowcast_Task2_opr_nrt13_0.49.pth, thresh

In [62]:
period_results = GenMetrics_lstm(
    model_name="lead6_Task1_sci_nrt",
    sample_obj=lead6_sci_nrt,
    model_dir="./LSTM_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-S-NRT-6.csv",
)

Threshold: 0.418[0.418,0.418]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 592
Bootstrap 1/30: lead6_Task1_sci_nrt0_0.4183333333333333.pth, threshold=0.418333
Bootstrap 2/30: lead6_Task1_sci_nrt1_0.4183333333333333.pth, threshold=0.418333
Bootstrap 3/30: lead6_Task1_sci_nrt2_0.4183333333333333.pth, threshold=0.418333
Bootstrap 4/30: lead6_Task1_sci_nrt3_0.4183333333333333.pth, threshold=0.418333
Bootstrap 5/30: lead6_Task1_sci_nrt4_0.4183333333333333.pth, threshold=0.418333
Bootstrap 6/30: lead6_Task1_sci_nrt5_0.4183333333333333.pth, threshold=0.418333
Bootstrap 7/30: lead6_Task1_sci_nrt6_0.4183333333333333.pth, threshold=0.418333
Bootstrap 8/30: lead6_Task1_sci_nrt7_0.4183333333333333.pth, threshold=0.418333
Bootstrap 9/30: lead6_Task1_sci_nrt8_0.4183333333333333.pth, threshold=0.418333
Bootstrap 10/30: lead6_Task1_sci_nrt9_0.4183333333333333.pth, threshold=0.418333
Bootstrap 11/30: lead6_Task1_sci_nrt10_0.4183333333333333.pth, threshold=0.418333
Bootstrap 12/

In [64]:
period_results = GenMetrics_lstm(
    model_name="lead6_Task2_sci_nrt",
    sample_obj=lead6_sci_nrt,
    model_dir="./LSTM_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-S-NRT-6.csv",
)

Threshold: 0.540[0.540,0.540]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 858
Bootstrap 1/30: lead6_Task2_sci_nrt0_0.54.pth, threshold=0.54
Bootstrap 2/30: lead6_Task2_sci_nrt1_0.54.pth, threshold=0.54
Bootstrap 3/30: lead6_Task2_sci_nrt2_0.54.pth, threshold=0.54
Bootstrap 4/30: lead6_Task2_sci_nrt3_0.54.pth, threshold=0.54
Bootstrap 5/30: lead6_Task2_sci_nrt4_0.54.pth, threshold=0.54
Bootstrap 6/30: lead6_Task2_sci_nrt5_0.54.pth, threshold=0.54
Bootstrap 7/30: lead6_Task2_sci_nrt6_0.54.pth, threshold=0.54
Bootstrap 8/30: lead6_Task2_sci_nrt7_0.54.pth, threshold=0.54
Bootstrap 9/30: lead6_Task2_sci_nrt8_0.54.pth, threshold=0.54
Bootstrap 10/30: lead6_Task2_sci_nrt9_0.54.pth, threshold=0.54
Bootstrap 11/30: lead6_Task2_sci_nrt10_0.54.pth, threshold=0.54
Bootstrap 12/30: lead6_Task2_sci_nrt11_0.54.pth, threshold=0.54
Bootstrap 13/30: lead6_Task2_sci_nrt12_0.54.pth, threshold=0.54
Bootstrap 14/30: lead6_Task2_sci_nrt13_0.54.pth, threshold=0.54
Bootstrap 15/30: le

In [33]:
period_results = GenMetrics_lstm(
    model_name="lead6_Task1_opr_nrt",
    sample_obj=lead6_opr_nrt,
    model_dir="./LSTM_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-O-NRT-6.csv",
)

Threshold: 0.440[0.440,0.440]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 619
Bootstrap 1/30: lead6_Task1_opr_nrt0_0.44.pth, threshold=0.44
Bootstrap 2/30: lead6_Task1_opr_nrt1_0.44.pth, threshold=0.44
Bootstrap 3/30: lead6_Task1_opr_nrt2_0.44.pth, threshold=0.44
Bootstrap 4/30: lead6_Task1_opr_nrt3_0.44.pth, threshold=0.44
Bootstrap 5/30: lead6_Task1_opr_nrt4_0.44.pth, threshold=0.44
Bootstrap 6/30: lead6_Task1_opr_nrt5_0.44.pth, threshold=0.44
Bootstrap 7/30: lead6_Task1_opr_nrt6_0.44.pth, threshold=0.44
Bootstrap 8/30: lead6_Task1_opr_nrt7_0.44.pth, threshold=0.44
Bootstrap 9/30: lead6_Task1_opr_nrt8_0.44.pth, threshold=0.44
Bootstrap 10/30: lead6_Task1_opr_nrt9_0.44.pth, threshold=0.44
Bootstrap 11/30: lead6_Task1_opr_nrt10_0.44.pth, threshold=0.44
Bootstrap 12/30: lead6_Task1_opr_nrt11_0.44.pth, threshold=0.44
Bootstrap 13/30: lead6_Task1_opr_nrt12_0.44.pth, threshold=0.44
Bootstrap 14/30: lead6_Task1_opr_nrt13_0.44.pth, threshold=0.44
Bootstrap 15/30: le

In [34]:
period_results = GenMetrics_lstm(
    model_name="lead6_Task2_opr_nrt",
    sample_obj=lead6_opr_nrt,
    model_dir="./LSTM_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-O-NRT-6.csv",
)

Threshold: 0.520[0.520,0.520]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 916
Bootstrap 1/30: lead6_Task2_opr_nrt0_0.52.pth, threshold=0.52
Bootstrap 2/30: lead6_Task2_opr_nrt1_0.52.pth, threshold=0.52
Bootstrap 3/30: lead6_Task2_opr_nrt2_0.52.pth, threshold=0.52
Bootstrap 4/30: lead6_Task2_opr_nrt3_0.52.pth, threshold=0.52
Bootstrap 5/30: lead6_Task2_opr_nrt4_0.52.pth, threshold=0.52
Bootstrap 6/30: lead6_Task2_opr_nrt5_0.52.pth, threshold=0.52
Bootstrap 7/30: lead6_Task2_opr_nrt6_0.52.pth, threshold=0.52
Bootstrap 8/30: lead6_Task2_opr_nrt7_0.52.pth, threshold=0.52
Bootstrap 9/30: lead6_Task2_opr_nrt8_0.52.pth, threshold=0.52
Bootstrap 10/30: lead6_Task2_opr_nrt9_0.52.pth, threshold=0.52
Bootstrap 11/30: lead6_Task2_opr_nrt10_0.52.pth, threshold=0.52
Bootstrap 12/30: lead6_Task2_opr_nrt11_0.52.pth, threshold=0.52
Bootstrap 13/30: lead6_Task2_opr_nrt12_0.52.pth, threshold=0.52
Bootstrap 14/30: lead6_Task2_opr_nrt13_0.52.pth, threshold=0.52
Bootstrap 15/30: le

In [35]:
period_results = GenMetrics_lstm(
    model_name="lead12_Task1_sci_nrt",
    sample_obj=lead12_sci_nrt,
    model_dir="./LSTM_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-S-NRT-12.csv",
)

Threshold: 0.450[0.450,0.450]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 574
Bootstrap 1/30: lead12_Task1_sci_nrt0_0.45.pth, threshold=0.45
Bootstrap 2/30: lead12_Task1_sci_nrt1_0.45.pth, threshold=0.45
Bootstrap 3/30: lead12_Task1_sci_nrt2_0.45.pth, threshold=0.45
Bootstrap 4/30: lead12_Task1_sci_nrt3_0.45.pth, threshold=0.45
Bootstrap 5/30: lead12_Task1_sci_nrt4_0.45.pth, threshold=0.45
Bootstrap 6/30: lead12_Task1_sci_nrt5_0.45.pth, threshold=0.45
Bootstrap 7/30: lead12_Task1_sci_nrt6_0.45.pth, threshold=0.45
Bootstrap 8/30: lead12_Task1_sci_nrt7_0.45.pth, threshold=0.45
Bootstrap 9/30: lead12_Task1_sci_nrt8_0.45.pth, threshold=0.45
Bootstrap 10/30: lead12_Task1_sci_nrt9_0.45.pth, threshold=0.45
Bootstrap 11/30: lead12_Task1_sci_nrt10_0.45.pth, threshold=0.45
Bootstrap 12/30: lead12_Task1_sci_nrt11_0.45.pth, threshold=0.45
Bootstrap 13/30: lead12_Task1_sci_nrt12_0.45.pth, threshold=0.45
Bootstrap 14/30: lead12_Task1_sci_nrt13_0.45.pth, threshold=0.45
Boots

In [36]:
period_results = GenMetrics_lstm(
    model_name="lead12_Task2_sci_nrt",
    sample_obj=lead12_sci_nrt,
    model_dir="./LSTM_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-S-NRT-12.csv",
)

Threshold: 0.440[0.440,0.440]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 829
Bootstrap 1/30: lead12_Task2_sci_nrt0_0.44.pth, threshold=0.44
Bootstrap 2/30: lead12_Task2_sci_nrt1_0.44.pth, threshold=0.44
Bootstrap 3/30: lead12_Task2_sci_nrt2_0.44.pth, threshold=0.44
Bootstrap 4/30: lead12_Task2_sci_nrt3_0.44.pth, threshold=0.44
Bootstrap 5/30: lead12_Task2_sci_nrt4_0.44.pth, threshold=0.44
Bootstrap 6/30: lead12_Task2_sci_nrt5_0.44.pth, threshold=0.44
Bootstrap 7/30: lead12_Task2_sci_nrt6_0.44.pth, threshold=0.44
Bootstrap 8/30: lead12_Task2_sci_nrt7_0.44.pth, threshold=0.44
Bootstrap 9/30: lead12_Task2_sci_nrt8_0.44.pth, threshold=0.44
Bootstrap 10/30: lead12_Task2_sci_nrt9_0.44.pth, threshold=0.44
Bootstrap 11/30: lead12_Task2_sci_nrt10_0.44.pth, threshold=0.44
Bootstrap 12/30: lead12_Task2_sci_nrt11_0.44.pth, threshold=0.44
Bootstrap 13/30: lead12_Task2_sci_nrt12_0.44.pth, threshold=0.44
Bootstrap 14/30: lead12_Task2_sci_nrt13_0.44.pth, threshold=0.44
Boots

In [37]:
period_results = GenMetrics_lstm(
    model_name="lead12_Task1_opr_nrt",
    sample_obj=lead12_opr_nrt,
    model_dir="./LSTM_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-O-NRT-12.csv",
)

Threshold: 0.523[0.523,0.523]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 607
Bootstrap 1/30: lead12_Task1_opr_nrt0_0.5233333333333333.pth, threshold=0.523333
Bootstrap 2/30: lead12_Task1_opr_nrt1_0.5233333333333333.pth, threshold=0.523333
Bootstrap 3/30: lead12_Task1_opr_nrt2_0.5233333333333333.pth, threshold=0.523333
Bootstrap 4/30: lead12_Task1_opr_nrt3_0.5233333333333333.pth, threshold=0.523333
Bootstrap 5/30: lead12_Task1_opr_nrt4_0.5233333333333333.pth, threshold=0.523333
Bootstrap 6/30: lead12_Task1_opr_nrt5_0.5233333333333333.pth, threshold=0.523333
Bootstrap 7/30: lead12_Task1_opr_nrt6_0.5233333333333333.pth, threshold=0.523333
Bootstrap 8/30: lead12_Task1_opr_nrt7_0.5233333333333333.pth, threshold=0.523333
Bootstrap 9/30: lead12_Task1_opr_nrt8_0.5233333333333333.pth, threshold=0.523333
Bootstrap 10/30: lead12_Task1_opr_nrt9_0.5233333333333333.pth, threshold=0.523333
Bootstrap 11/30: lead12_Task1_opr_nrt10_0.5233333333333333.pth, threshold=0.523333
Bo

In [38]:
period_results = GenMetrics_lstm(
    model_name="lead12_Task2_opr_nrt",
    sample_obj=lead12_opr_nrt,
    model_dir="./LSTM_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-O-NRT-12.csv",
)

Threshold: 0.540[0.540,0.540]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 890
Bootstrap 1/30: lead12_Task2_opr_nrt0_0.54.pth, threshold=0.54
Bootstrap 2/30: lead12_Task2_opr_nrt1_0.54.pth, threshold=0.54
Bootstrap 3/30: lead12_Task2_opr_nrt2_0.54.pth, threshold=0.54
Bootstrap 4/30: lead12_Task2_opr_nrt3_0.54.pth, threshold=0.54
Bootstrap 5/30: lead12_Task2_opr_nrt4_0.54.pth, threshold=0.54
Bootstrap 6/30: lead12_Task2_opr_nrt5_0.54.pth, threshold=0.54
Bootstrap 7/30: lead12_Task2_opr_nrt6_0.54.pth, threshold=0.54
Bootstrap 8/30: lead12_Task2_opr_nrt7_0.54.pth, threshold=0.54
Bootstrap 9/30: lead12_Task2_opr_nrt8_0.54.pth, threshold=0.54
Bootstrap 10/30: lead12_Task2_opr_nrt9_0.54.pth, threshold=0.54
Bootstrap 11/30: lead12_Task2_opr_nrt10_0.54.pth, threshold=0.54
Bootstrap 12/30: lead12_Task2_opr_nrt11_0.54.pth, threshold=0.54
Bootstrap 13/30: lead12_Task2_opr_nrt12_0.54.pth, threshold=0.54
Bootstrap 14/30: lead12_Task2_opr_nrt13_0.54.pth, threshold=0.54
Boots

In [39]:
period_results = GenMetrics_lstm(
    model_name="lead24_Task1_sci_nrt",
    sample_obj=lead24_sci_nrt,
    model_dir="./LSTM_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-S-NRT-24.csv",
)

Threshold: 0.500[0.500,0.500]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 536
Bootstrap 1/30: lead24_Task1_sci_nrt0_0.5000000000000001.pth, threshold=0.5
Bootstrap 2/30: lead24_Task1_sci_nrt1_0.5000000000000001.pth, threshold=0.5
Bootstrap 3/30: lead24_Task1_sci_nrt2_0.5000000000000001.pth, threshold=0.5
Bootstrap 4/30: lead24_Task1_sci_nrt3_0.5000000000000001.pth, threshold=0.5
Bootstrap 5/30: lead24_Task1_sci_nrt4_0.5000000000000001.pth, threshold=0.5
Bootstrap 6/30: lead24_Task1_sci_nrt5_0.5000000000000001.pth, threshold=0.5
Bootstrap 7/30: lead24_Task1_sci_nrt6_0.5000000000000001.pth, threshold=0.5
Bootstrap 8/30: lead24_Task1_sci_nrt7_0.5000000000000001.pth, threshold=0.5
Bootstrap 9/30: lead24_Task1_sci_nrt8_0.5000000000000001.pth, threshold=0.5
Bootstrap 10/30: lead24_Task1_sci_nrt9_0.5000000000000001.pth, threshold=0.5
Bootstrap 11/30: lead24_Task1_sci_nrt10_0.5000000000000001.pth, threshold=0.5
Bootstrap 12/30: lead24_Task1_sci_nrt11_0.500000000000000

In [40]:
period_results = GenMetrics_lstm(
    model_name="lead24_Task2_sci_nrt",
    sample_obj=lead24_sci_nrt,
    model_dir="./LSTM_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-S-NRT-24.csv",
)

Threshold: 0.540[0.540,0.540]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 762
Bootstrap 1/30: lead24_Task2_sci_nrt0_0.54.pth, threshold=0.54
Bootstrap 2/30: lead24_Task2_sci_nrt1_0.54.pth, threshold=0.54
Bootstrap 3/30: lead24_Task2_sci_nrt2_0.54.pth, threshold=0.54
Bootstrap 4/30: lead24_Task2_sci_nrt3_0.54.pth, threshold=0.54
Bootstrap 5/30: lead24_Task2_sci_nrt4_0.54.pth, threshold=0.54
Bootstrap 6/30: lead24_Task2_sci_nrt5_0.54.pth, threshold=0.54
Bootstrap 7/30: lead24_Task2_sci_nrt6_0.54.pth, threshold=0.54
Bootstrap 8/30: lead24_Task2_sci_nrt7_0.54.pth, threshold=0.54
Bootstrap 9/30: lead24_Task2_sci_nrt8_0.54.pth, threshold=0.54
Bootstrap 10/30: lead24_Task2_sci_nrt9_0.54.pth, threshold=0.54
Bootstrap 11/30: lead24_Task2_sci_nrt10_0.54.pth, threshold=0.54
Bootstrap 12/30: lead24_Task2_sci_nrt11_0.54.pth, threshold=0.54
Bootstrap 13/30: lead24_Task2_sci_nrt12_0.54.pth, threshold=0.54
Bootstrap 14/30: lead24_Task2_sci_nrt13_0.54.pth, threshold=0.54
Boots

In [41]:
period_results = GenMetrics_lstm(
    model_name="lead24_Task1_opr_nrt",
    sample_obj=lead24_opr_nrt,
    model_dir="./LSTM_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_LSTM-O-NRT-24.csv",
)

Threshold: 0.433[0.433,0.433]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 589
Bootstrap 1/30: lead24_Task1_opr_nrt0_0.43333333333333335.pth, threshold=0.433333
Bootstrap 2/30: lead24_Task1_opr_nrt1_0.43333333333333335.pth, threshold=0.433333
Bootstrap 3/30: lead24_Task1_opr_nrt2_0.43333333333333335.pth, threshold=0.433333
Bootstrap 4/30: lead24_Task1_opr_nrt3_0.43333333333333335.pth, threshold=0.433333
Bootstrap 5/30: lead24_Task1_opr_nrt4_0.43333333333333335.pth, threshold=0.433333
Bootstrap 6/30: lead24_Task1_opr_nrt5_0.43333333333333335.pth, threshold=0.433333
Bootstrap 7/30: lead24_Task1_opr_nrt6_0.43333333333333335.pth, threshold=0.433333
Bootstrap 8/30: lead24_Task1_opr_nrt7_0.43333333333333335.pth, threshold=0.433333
Bootstrap 9/30: lead24_Task1_opr_nrt8_0.43333333333333335.pth, threshold=0.433333
Bootstrap 10/30: lead24_Task1_opr_nrt9_0.43333333333333335.pth, threshold=0.433333
Bootstrap 11/30: lead24_Task1_opr_nrt10_0.43333333333333335.pth, threshold=

In [42]:
period_results = GenMetrics_lstm(
    model_name="lead24_Task2_opr_nrt",
    sample_obj=lead24_opr_nrt,
    model_dir="./LSTM_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_LSTM-O-NRT-24.csv",
)

Threshold: 0.500[0.500,0.500]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 837
Bootstrap 1/30: lead24_Task2_opr_nrt0_0.5.pth, threshold=0.5
Bootstrap 2/30: lead24_Task2_opr_nrt1_0.5.pth, threshold=0.5
Bootstrap 3/30: lead24_Task2_opr_nrt2_0.5.pth, threshold=0.5
Bootstrap 4/30: lead24_Task2_opr_nrt3_0.5.pth, threshold=0.5
Bootstrap 5/30: lead24_Task2_opr_nrt4_0.5.pth, threshold=0.5
Bootstrap 6/30: lead24_Task2_opr_nrt5_0.5.pth, threshold=0.5
Bootstrap 7/30: lead24_Task2_opr_nrt6_0.5.pth, threshold=0.5
Bootstrap 8/30: lead24_Task2_opr_nrt7_0.5.pth, threshold=0.5
Bootstrap 9/30: lead24_Task2_opr_nrt8_0.5.pth, threshold=0.5
Bootstrap 10/30: lead24_Task2_opr_nrt9_0.5.pth, threshold=0.5
Bootstrap 11/30: lead24_Task2_opr_nrt10_0.5.pth, threshold=0.5
Bootstrap 12/30: lead24_Task2_opr_nrt11_0.5.pth, threshold=0.5
Bootstrap 13/30: lead24_Task2_opr_nrt12_0.5.pth, threshold=0.5
Bootstrap 14/30: lead24_Task2_opr_nrt13_0.5.pth, threshold=0.5
Bootstrap 15/30: lead24_Task2_opr

### Logreg + NRT

In [43]:
from Gen_Results import GenMetrics_logreg

#importlib.reload(Gen_Results)

In [44]:
period_results = GenMetrics_logreg(
    model_name="logistic_nowcast_Task1_sci_nrt",
    sample_obj=nowcast_sci_nrt,
    model_dir="./Logreg_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-S-NRT-0.csv",
)

Threshold: 0.490[0.490,0.490]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 602
Bootstrap 1/30: logistic_nowcast_Task1_sci_nrt0_0.49.pth, threshold=0.49
Bootstrap 2/30: logistic_nowcast_Task1_sci_nrt1_0.49.pth, threshold=0.49
Bootstrap 3/30: logistic_nowcast_Task1_sci_nrt2_0.49.pth, threshold=0.49
Bootstrap 4/30: logistic_nowcast_Task1_sci_nrt3_0.49.pth, threshold=0.49
Bootstrap 5/30: logistic_nowcast_Task1_sci_nrt4_0.49.pth, threshold=0.49
Bootstrap 6/30: logistic_nowcast_Task1_sci_nrt5_0.49.pth, threshold=0.49
Bootstrap 7/30: logistic_nowcast_Task1_sci_nrt6_0.49.pth, threshold=0.49
Bootstrap 8/30: logistic_nowcast_Task1_sci_nrt7_0.49.pth, threshold=0.49
Bootstrap 9/30: logistic_nowcast_Task1_sci_nrt8_0.49.pth, threshold=0.49
Bootstrap 10/30: logistic_nowcast_Task1_sci_nrt9_0.49.pth, threshold=0.49
Bootstrap 11/30: logistic_nowcast_Task1_sci_nrt10_0.49.pth, threshold=0.49
Bootstrap 12/30: logistic_nowcast_Task1_sci_nrt11_0.49.pth, threshold=0.49
Bootstrap 13/30

In [45]:
period_results = GenMetrics_logreg(
    model_name="logistic_nowcast_Task2_sci_nrt",
    sample_obj=nowcast_sci_nrt,
    model_dir="./Logreg_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-S-NRT-0.csv",
)

Threshold: 0.350[0.350,0.350]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 879
Bootstrap 1/30: logistic_nowcast_Task2_sci_nrt0_0.35.pth, threshold=0.35
Bootstrap 2/30: logistic_nowcast_Task2_sci_nrt1_0.35.pth, threshold=0.35
Bootstrap 3/30: logistic_nowcast_Task2_sci_nrt2_0.35.pth, threshold=0.35
Bootstrap 4/30: logistic_nowcast_Task2_sci_nrt3_0.35.pth, threshold=0.35
Bootstrap 5/30: logistic_nowcast_Task2_sci_nrt4_0.35.pth, threshold=0.35
Bootstrap 6/30: logistic_nowcast_Task2_sci_nrt5_0.35.pth, threshold=0.35
Bootstrap 7/30: logistic_nowcast_Task2_sci_nrt6_0.35.pth, threshold=0.35
Bootstrap 8/30: logistic_nowcast_Task2_sci_nrt7_0.35.pth, threshold=0.35
Bootstrap 9/30: logistic_nowcast_Task2_sci_nrt8_0.35.pth, threshold=0.35
Bootstrap 10/30: logistic_nowcast_Task2_sci_nrt9_0.35.pth, threshold=0.35
Bootstrap 11/30: logistic_nowcast_Task2_sci_nrt10_0.35.pth, threshold=0.35
Bootstrap 12/30: logistic_nowcast_Task2_sci_nrt11_0.35.pth, threshold=0.35
Bootstrap 13/30

In [46]:
period_results = GenMetrics_logreg(
    model_name="logistic_nowcast_Task1_opr_nrt",
    sample_obj=nowcast_opr_nrt,
    model_dir="./Logreg_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-O-NRT-0.csv",
)

Threshold: 0.540[0.540,0.540]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 623
Bootstrap 1/30: logistic_nowcast_Task1_opr_nrt0_0.54.pth, threshold=0.54
Bootstrap 2/30: logistic_nowcast_Task1_opr_nrt1_0.54.pth, threshold=0.54
Bootstrap 3/30: logistic_nowcast_Task1_opr_nrt2_0.54.pth, threshold=0.54
Bootstrap 4/30: logistic_nowcast_Task1_opr_nrt3_0.54.pth, threshold=0.54
Bootstrap 5/30: logistic_nowcast_Task1_opr_nrt4_0.54.pth, threshold=0.54
Bootstrap 6/30: logistic_nowcast_Task1_opr_nrt5_0.54.pth, threshold=0.54
Bootstrap 7/30: logistic_nowcast_Task1_opr_nrt6_0.54.pth, threshold=0.54
Bootstrap 8/30: logistic_nowcast_Task1_opr_nrt7_0.54.pth, threshold=0.54
Bootstrap 9/30: logistic_nowcast_Task1_opr_nrt8_0.54.pth, threshold=0.54
Bootstrap 10/30: logistic_nowcast_Task1_opr_nrt9_0.54.pth, threshold=0.54
Bootstrap 11/30: logistic_nowcast_Task1_opr_nrt10_0.54.pth, threshold=0.54
Bootstrap 12/30: logistic_nowcast_Task1_opr_nrt11_0.54.pth, threshold=0.54
Bootstrap 13/30

In [47]:
period_results = GenMetrics_logreg(
    model_name="logistic_nowcast_Task2_opr_nrt",
    sample_obj=nowcast_opr_nrt,
    model_dir="./Logreg_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-O-NRT-0.csv",
)

Threshold: 0.450[0.450,0.450]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 935
Bootstrap 1/30: logistic_nowcast_Task2_opr_nrt0_0.45.pth, threshold=0.45
Bootstrap 2/30: logistic_nowcast_Task2_opr_nrt1_0.45.pth, threshold=0.45
Bootstrap 3/30: logistic_nowcast_Task2_opr_nrt2_0.45.pth, threshold=0.45
Bootstrap 4/30: logistic_nowcast_Task2_opr_nrt3_0.45.pth, threshold=0.45
Bootstrap 5/30: logistic_nowcast_Task2_opr_nrt4_0.45.pth, threshold=0.45
Bootstrap 6/30: logistic_nowcast_Task2_opr_nrt5_0.45.pth, threshold=0.45
Bootstrap 7/30: logistic_nowcast_Task2_opr_nrt6_0.45.pth, threshold=0.45
Bootstrap 8/30: logistic_nowcast_Task2_opr_nrt7_0.45.pth, threshold=0.45
Bootstrap 9/30: logistic_nowcast_Task2_opr_nrt8_0.45.pth, threshold=0.45
Bootstrap 10/30: logistic_nowcast_Task2_opr_nrt9_0.45.pth, threshold=0.45
Bootstrap 11/30: logistic_nowcast_Task2_opr_nrt10_0.45.pth, threshold=0.45
Bootstrap 12/30: logistic_nowcast_Task2_opr_nrt11_0.45.pth, threshold=0.45
Bootstrap 13/30

In [48]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead6_Task1_sci_nrt",
    sample_obj=lead6_sci_nrt,
    model_dir="./Logreg_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-S-NRT-6.csv",
)

Threshold: 0.690[0.690,0.690]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 592
Bootstrap 1/30: logistic_lead6_Task1_sci_nrt0_0.69.pth, threshold=0.69
Bootstrap 2/30: logistic_lead6_Task1_sci_nrt1_0.69.pth, threshold=0.69
Bootstrap 3/30: logistic_lead6_Task1_sci_nrt2_0.69.pth, threshold=0.69
Bootstrap 4/30: logistic_lead6_Task1_sci_nrt3_0.69.pth, threshold=0.69
Bootstrap 5/30: logistic_lead6_Task1_sci_nrt4_0.69.pth, threshold=0.69
Bootstrap 6/30: logistic_lead6_Task1_sci_nrt5_0.69.pth, threshold=0.69
Bootstrap 7/30: logistic_lead6_Task1_sci_nrt6_0.69.pth, threshold=0.69
Bootstrap 8/30: logistic_lead6_Task1_sci_nrt7_0.69.pth, threshold=0.69
Bootstrap 9/30: logistic_lead6_Task1_sci_nrt8_0.69.pth, threshold=0.69
Bootstrap 10/30: logistic_lead6_Task1_sci_nrt9_0.69.pth, threshold=0.69
Bootstrap 11/30: logistic_lead6_Task1_sci_nrt10_0.69.pth, threshold=0.69
Bootstrap 12/30: logistic_lead6_Task1_sci_nrt11_0.69.pth, threshold=0.69
Bootstrap 13/30: logistic_lead6_Task1_s

In [49]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead6_Task2_sci_nrt",
    sample_obj=lead6_sci_nrt,
    model_dir="./Logreg_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-S-NRT-6.csv",
)

Threshold: 0.450[0.450,0.450]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 858
Bootstrap 1/30: logistic_lead6_Task2_sci_nrt0_0.45.pth, threshold=0.45
Bootstrap 2/30: logistic_lead6_Task2_sci_nrt1_0.45.pth, threshold=0.45
Bootstrap 3/30: logistic_lead6_Task2_sci_nrt2_0.45.pth, threshold=0.45
Bootstrap 4/30: logistic_lead6_Task2_sci_nrt3_0.45.pth, threshold=0.45
Bootstrap 5/30: logistic_lead6_Task2_sci_nrt4_0.45.pth, threshold=0.45
Bootstrap 6/30: logistic_lead6_Task2_sci_nrt5_0.45.pth, threshold=0.45
Bootstrap 7/30: logistic_lead6_Task2_sci_nrt6_0.45.pth, threshold=0.45
Bootstrap 8/30: logistic_lead6_Task2_sci_nrt7_0.45.pth, threshold=0.45
Bootstrap 9/30: logistic_lead6_Task2_sci_nrt8_0.45.pth, threshold=0.45
Bootstrap 10/30: logistic_lead6_Task2_sci_nrt9_0.45.pth, threshold=0.45
Bootstrap 11/30: logistic_lead6_Task2_sci_nrt10_0.45.pth, threshold=0.45
Bootstrap 12/30: logistic_lead6_Task2_sci_nrt11_0.45.pth, threshold=0.45
Bootstrap 13/30: logistic_lead6_Task2_s

In [50]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead6_Task1_opr_nrt",
    sample_obj=lead6_opr_nrt,
    model_dir="./Logreg_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-O-NRT-6.csv",
)

Threshold: 0.570[0.570,0.570]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 619
Bootstrap 1/30: logistic_lead6_Task1_opr_nrt0_0.57.pth, threshold=0.57
Bootstrap 2/30: logistic_lead6_Task1_opr_nrt1_0.57.pth, threshold=0.57
Bootstrap 3/30: logistic_lead6_Task1_opr_nrt2_0.57.pth, threshold=0.57
Bootstrap 4/30: logistic_lead6_Task1_opr_nrt3_0.57.pth, threshold=0.57
Bootstrap 5/30: logistic_lead6_Task1_opr_nrt4_0.57.pth, threshold=0.57
Bootstrap 6/30: logistic_lead6_Task1_opr_nrt5_0.57.pth, threshold=0.57
Bootstrap 7/30: logistic_lead6_Task1_opr_nrt6_0.57.pth, threshold=0.57
Bootstrap 8/30: logistic_lead6_Task1_opr_nrt7_0.57.pth, threshold=0.57
Bootstrap 9/30: logistic_lead6_Task1_opr_nrt8_0.57.pth, threshold=0.57
Bootstrap 10/30: logistic_lead6_Task1_opr_nrt9_0.57.pth, threshold=0.57
Bootstrap 11/30: logistic_lead6_Task1_opr_nrt10_0.57.pth, threshold=0.57
Bootstrap 12/30: logistic_lead6_Task1_opr_nrt11_0.57.pth, threshold=0.57
Bootstrap 13/30: logistic_lead6_Task1_o

In [65]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead6_Task2_opr_nrt",
    sample_obj=lead6_opr_nrt,
    model_dir="./Logreg_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-O-NRT-6.csv",
)

Threshold: 0.410[0.410,0.410]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 916
Bootstrap 1/30: logistic_lead6_Task2_opr_nrt0_0.41.pth, threshold=0.41
Bootstrap 2/30: logistic_lead6_Task2_opr_nrt1_0.41.pth, threshold=0.41
Bootstrap 3/30: logistic_lead6_Task2_opr_nrt2_0.41.pth, threshold=0.41
Bootstrap 4/30: logistic_lead6_Task2_opr_nrt3_0.41.pth, threshold=0.41
Bootstrap 5/30: logistic_lead6_Task2_opr_nrt4_0.41.pth, threshold=0.41
Bootstrap 6/30: logistic_lead6_Task2_opr_nrt5_0.41.pth, threshold=0.41
Bootstrap 7/30: logistic_lead6_Task2_opr_nrt6_0.41.pth, threshold=0.41
Bootstrap 8/30: logistic_lead6_Task2_opr_nrt7_0.41.pth, threshold=0.41
Bootstrap 9/30: logistic_lead6_Task2_opr_nrt8_0.41.pth, threshold=0.41
Bootstrap 10/30: logistic_lead6_Task2_opr_nrt9_0.41.pth, threshold=0.41
Bootstrap 11/30: logistic_lead6_Task2_opr_nrt10_0.41.pth, threshold=0.41
Bootstrap 12/30: logistic_lead6_Task2_opr_nrt11_0.41.pth, threshold=0.41
Bootstrap 13/30: logistic_lead6_Task2_o

In [52]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead12_Task1_sci_nrt",
    sample_obj=lead12_sci_nrt,
    model_dir="./Logreg_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-S-NRT-12.csv",
)

Threshold: 0.390[0.390,0.390]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 574
Bootstrap 1/30: logistic_lead12_Task1_sci_nrt0_0.39.pth, threshold=0.39
Bootstrap 2/30: logistic_lead12_Task1_sci_nrt1_0.39.pth, threshold=0.39
Bootstrap 3/30: logistic_lead12_Task1_sci_nrt2_0.39.pth, threshold=0.39
Bootstrap 4/30: logistic_lead12_Task1_sci_nrt3_0.39.pth, threshold=0.39
Bootstrap 5/30: logistic_lead12_Task1_sci_nrt4_0.39.pth, threshold=0.39
Bootstrap 6/30: logistic_lead12_Task1_sci_nrt5_0.39.pth, threshold=0.39
Bootstrap 7/30: logistic_lead12_Task1_sci_nrt6_0.39.pth, threshold=0.39
Bootstrap 8/30: logistic_lead12_Task1_sci_nrt7_0.39.pth, threshold=0.39
Bootstrap 9/30: logistic_lead12_Task1_sci_nrt8_0.39.pth, threshold=0.39
Bootstrap 10/30: logistic_lead12_Task1_sci_nrt9_0.39.pth, threshold=0.39
Bootstrap 11/30: logistic_lead12_Task1_sci_nrt10_0.39.pth, threshold=0.39
Bootstrap 12/30: logistic_lead12_Task1_sci_nrt11_0.39.pth, threshold=0.39
Bootstrap 13/30: logistic_l

In [53]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead12_Task2_sci_nrt",
    sample_obj=lead12_sci_nrt,
    model_dir="./Logreg_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-S-NRT-12.csv",
)

Threshold: 0.520[0.520,0.520]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 829
Bootstrap 1/30: logistic_lead12_Task2_sci_nrt0_0.52.pth, threshold=0.52
Bootstrap 2/30: logistic_lead12_Task2_sci_nrt1_0.52.pth, threshold=0.52
Bootstrap 3/30: logistic_lead12_Task2_sci_nrt2_0.52.pth, threshold=0.52
Bootstrap 4/30: logistic_lead12_Task2_sci_nrt3_0.52.pth, threshold=0.52
Bootstrap 5/30: logistic_lead12_Task2_sci_nrt4_0.52.pth, threshold=0.52
Bootstrap 6/30: logistic_lead12_Task2_sci_nrt5_0.52.pth, threshold=0.52
Bootstrap 7/30: logistic_lead12_Task2_sci_nrt6_0.52.pth, threshold=0.52
Bootstrap 8/30: logistic_lead12_Task2_sci_nrt7_0.52.pth, threshold=0.52
Bootstrap 9/30: logistic_lead12_Task2_sci_nrt8_0.52.pth, threshold=0.52
Bootstrap 10/30: logistic_lead12_Task2_sci_nrt9_0.52.pth, threshold=0.52
Bootstrap 11/30: logistic_lead12_Task2_sci_nrt10_0.52.pth, threshold=0.52
Bootstrap 12/30: logistic_lead12_Task2_sci_nrt11_0.52.pth, threshold=0.52
Bootstrap 13/30: logistic_l

In [54]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead12_Task1_opr_nrt",
    sample_obj=lead12_opr_nrt,
    model_dir="./Logreg_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-O-NRT-12.csv",
)

Threshold: 0.520[0.520,0.520]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 607
Bootstrap 1/30: logistic_lead12_Task1_opr_nrt0_0.52.pth, threshold=0.52
Bootstrap 2/30: logistic_lead12_Task1_opr_nrt1_0.52.pth, threshold=0.52
Bootstrap 3/30: logistic_lead12_Task1_opr_nrt2_0.52.pth, threshold=0.52
Bootstrap 4/30: logistic_lead12_Task1_opr_nrt3_0.52.pth, threshold=0.52
Bootstrap 5/30: logistic_lead12_Task1_opr_nrt4_0.52.pth, threshold=0.52
Bootstrap 6/30: logistic_lead12_Task1_opr_nrt5_0.52.pth, threshold=0.52
Bootstrap 7/30: logistic_lead12_Task1_opr_nrt6_0.52.pth, threshold=0.52
Bootstrap 8/30: logistic_lead12_Task1_opr_nrt7_0.52.pth, threshold=0.52
Bootstrap 9/30: logistic_lead12_Task1_opr_nrt8_0.52.pth, threshold=0.52
Bootstrap 10/30: logistic_lead12_Task1_opr_nrt9_0.52.pth, threshold=0.52
Bootstrap 11/30: logistic_lead12_Task1_opr_nrt10_0.52.pth, threshold=0.52
Bootstrap 12/30: logistic_lead12_Task1_opr_nrt11_0.52.pth, threshold=0.52
Bootstrap 13/30: logistic_l

In [55]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead12_Task2_opr_nrt",
    sample_obj=lead12_opr_nrt,
    model_dir="./Logreg_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-O-NRT-12.csv",
)

Threshold: 0.410[0.410,0.410]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 890
Bootstrap 1/30: logistic_lead12_Task2_opr_nrt0_0.41.pth, threshold=0.41
Bootstrap 2/30: logistic_lead12_Task2_opr_nrt1_0.41.pth, threshold=0.41
Bootstrap 3/30: logistic_lead12_Task2_opr_nrt2_0.41.pth, threshold=0.41
Bootstrap 4/30: logistic_lead12_Task2_opr_nrt3_0.41.pth, threshold=0.41
Bootstrap 5/30: logistic_lead12_Task2_opr_nrt4_0.41.pth, threshold=0.41
Bootstrap 6/30: logistic_lead12_Task2_opr_nrt5_0.41.pth, threshold=0.41
Bootstrap 7/30: logistic_lead12_Task2_opr_nrt6_0.41.pth, threshold=0.41
Bootstrap 8/30: logistic_lead12_Task2_opr_nrt7_0.41.pth, threshold=0.41
Bootstrap 9/30: logistic_lead12_Task2_opr_nrt8_0.41.pth, threshold=0.41
Bootstrap 10/30: logistic_lead12_Task2_opr_nrt9_0.41.pth, threshold=0.41
Bootstrap 11/30: logistic_lead12_Task2_opr_nrt10_0.41.pth, threshold=0.41
Bootstrap 12/30: logistic_lead12_Task2_opr_nrt11_0.41.pth, threshold=0.41
Bootstrap 13/30: logistic_l

In [56]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead24_Task1_sci_nrt",
    sample_obj=lead24_sci_nrt,
    model_dir="./Logreg_models/Task1_sci_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-S-NRT-24.csv",
)

Threshold: 0.450[0.450,0.450]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 536
Bootstrap 1/30: logistic_lead24_Task1_sci_nrt0_0.45.pth, threshold=0.45
Bootstrap 2/30: logistic_lead24_Task1_sci_nrt1_0.45.pth, threshold=0.45
Bootstrap 3/30: logistic_lead24_Task1_sci_nrt2_0.45.pth, threshold=0.45
Bootstrap 4/30: logistic_lead24_Task1_sci_nrt3_0.45.pth, threshold=0.45
Bootstrap 5/30: logistic_lead24_Task1_sci_nrt4_0.45.pth, threshold=0.45
Bootstrap 6/30: logistic_lead24_Task1_sci_nrt5_0.45.pth, threshold=0.45
Bootstrap 7/30: logistic_lead24_Task1_sci_nrt6_0.45.pth, threshold=0.45
Bootstrap 8/30: logistic_lead24_Task1_sci_nrt7_0.45.pth, threshold=0.45
Bootstrap 9/30: logistic_lead24_Task1_sci_nrt8_0.45.pth, threshold=0.45
Bootstrap 10/30: logistic_lead24_Task1_sci_nrt9_0.45.pth, threshold=0.45
Bootstrap 11/30: logistic_lead24_Task1_sci_nrt10_0.45.pth, threshold=0.45
Bootstrap 12/30: logistic_lead24_Task1_sci_nrt11_0.45.pth, threshold=0.45
Bootstrap 13/30: logistic_l

In [57]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead24_Task2_sci_nrt",
    sample_obj=lead24_sci_nrt,
    model_dir="./Logreg_models/Task2_sci_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-S-NRT-24.csv",
)

Threshold: 0.480[0.480,0.480]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 39, Negative: 762
Bootstrap 1/30: logistic_lead24_Task2_sci_nrt0_0.48.pth, threshold=0.48
Bootstrap 2/30: logistic_lead24_Task2_sci_nrt1_0.48.pth, threshold=0.48
Bootstrap 3/30: logistic_lead24_Task2_sci_nrt2_0.48.pth, threshold=0.48
Bootstrap 4/30: logistic_lead24_Task2_sci_nrt3_0.48.pth, threshold=0.48
Bootstrap 5/30: logistic_lead24_Task2_sci_nrt4_0.48.pth, threshold=0.48
Bootstrap 6/30: logistic_lead24_Task2_sci_nrt5_0.48.pth, threshold=0.48
Bootstrap 7/30: logistic_lead24_Task2_sci_nrt6_0.48.pth, threshold=0.48
Bootstrap 8/30: logistic_lead24_Task2_sci_nrt7_0.48.pth, threshold=0.48
Bootstrap 9/30: logistic_lead24_Task2_sci_nrt8_0.48.pth, threshold=0.48
Bootstrap 10/30: logistic_lead24_Task2_sci_nrt9_0.48.pth, threshold=0.48
Bootstrap 11/30: logistic_lead24_Task2_sci_nrt10_0.48.pth, threshold=0.48
Bootstrap 12/30: logistic_lead24_Task2_sci_nrt11_0.48.pth, threshold=0.48
Bootstrap 13/30: logistic_l

In [58]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead24_Task1_opr_nrt",
    sample_obj=lead24_opr_nrt,
    model_dir="./Logreg_models/Task1_opr_nrt_models",
    purpose="Task1",
    results_filename="Task1_raw_metrics_Logreg-O-NRT-24.csv",
)

Threshold: 0.520[0.520,0.520]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 589
Bootstrap 1/30: logistic_lead24_Task1_opr_nrt0_0.52.pth, threshold=0.52
Bootstrap 2/30: logistic_lead24_Task1_opr_nrt1_0.52.pth, threshold=0.52
Bootstrap 3/30: logistic_lead24_Task1_opr_nrt2_0.52.pth, threshold=0.52
Bootstrap 4/30: logistic_lead24_Task1_opr_nrt3_0.52.pth, threshold=0.52
Bootstrap 5/30: logistic_lead24_Task1_opr_nrt4_0.52.pth, threshold=0.52
Bootstrap 6/30: logistic_lead24_Task1_opr_nrt5_0.52.pth, threshold=0.52
Bootstrap 7/30: logistic_lead24_Task1_opr_nrt6_0.52.pth, threshold=0.52
Bootstrap 8/30: logistic_lead24_Task1_opr_nrt7_0.52.pth, threshold=0.52
Bootstrap 9/30: logistic_lead24_Task1_opr_nrt8_0.52.pth, threshold=0.52
Bootstrap 10/30: logistic_lead24_Task1_opr_nrt9_0.52.pth, threshold=0.52
Bootstrap 11/30: logistic_lead24_Task1_opr_nrt10_0.52.pth, threshold=0.52
Bootstrap 12/30: logistic_lead24_Task1_opr_nrt11_0.52.pth, threshold=0.52
Bootstrap 13/30: logistic_l

In [59]:
period_results = GenMetrics_logreg(
    model_name="logistic_lead24_Task2_opr_nrt",
    sample_obj=lead24_opr_nrt,
    model_dir="./Logreg_models/Task2_opr_nrt_models",
    purpose="Task2",
    results_filename="Task2_raw_metrics_Logreg-O-NRT-24.csv",
)

Threshold: 0.460[0.460,0.460]

Testing period 1/3: 2020-01-01 to 2022-01-01
Positive: 73, Negative: 837
Bootstrap 1/30: logistic_lead24_Task2_opr_nrt0_0.46.pth, threshold=0.46
Bootstrap 2/30: logistic_lead24_Task2_opr_nrt1_0.46.pth, threshold=0.46
Bootstrap 3/30: logistic_lead24_Task2_opr_nrt2_0.46.pth, threshold=0.46
Bootstrap 4/30: logistic_lead24_Task2_opr_nrt3_0.46.pth, threshold=0.46
Bootstrap 5/30: logistic_lead24_Task2_opr_nrt4_0.46.pth, threshold=0.46
Bootstrap 6/30: logistic_lead24_Task2_opr_nrt5_0.46.pth, threshold=0.46
Bootstrap 7/30: logistic_lead24_Task2_opr_nrt6_0.46.pth, threshold=0.46
Bootstrap 8/30: logistic_lead24_Task2_opr_nrt7_0.46.pth, threshold=0.46
Bootstrap 9/30: logistic_lead24_Task2_opr_nrt8_0.46.pth, threshold=0.46
Bootstrap 10/30: logistic_lead24_Task2_opr_nrt9_0.46.pth, threshold=0.46
Bootstrap 11/30: logistic_lead24_Task2_opr_nrt10_0.46.pth, threshold=0.46
Bootstrap 12/30: logistic_lead24_Task2_opr_nrt11_0.46.pth, threshold=0.46
Bootstrap 13/30: logistic_l